# BatikCraft — LoRA Gaya Batik SDXL untuk Objek Apa Pun

Melatih **LoRA gaya** di atas Stable Diffusion XL sehingga BatikBrew dapat mengubah
foto objek apa pun (botol, bunga, wayang, kendaraan) menjadi ornamen batik.

**Prinsipnya:** dataset berisi **gambar batik saja**. Objek target tidak perlu ada di
dataset — bentuk objek disuplai saat inferensi (img2img + ControlNet), sedangkan LoRA
hanya belajar *gaya*: canting, isen-isen, palet soga/indigo, kontur ornamen.

| | SD 1.5 (notebook lama) | SDXL (notebook ini) |
|---|---|---|
| Resolusi | 512 px | 1024 px |
| Dipakai oleh | Batifikasi Objek | **BatikBrew** |
| VRAM latih | ~8 GB | ~15 GB (T4 ×2 / P100) |

Hasil akhir: paket `.batikmodel` yang langsung dapat dipasang di
**Pusat Dependensi → Model AI Offline & LoRA → Pasang .batikmodel**.

## Cara pakai di Kaggle
1. **Settings → Accelerator → GPU T4 ×2** (atau P100).
2. **Settings → Internet → On**.
3. Tambahkan dataset gambar batik sebagai *Kaggle Dataset* (folder berisi JPG/PNG).
4. Ubah `CFG.dataset_root` pada sel konfigurasi, lalu **Run All**.

In [ ]:
# 1) Dependensi Kaggle.
#    Dua jebakan lingkungan Kaggle yang ditangani di sini:
#    (a) diffusers 0.39 mensyaratkan peft >= 0.17;
#    (b) Kaggle memasang torchao lama (0.10) yang membuat PEFT melempar
#        ImportError saat LoRA di-inject. torchao tidak diperlukan untuk
#        pelatihan ini, jadi versi lama cukup dihapus.
import importlib, importlib.metadata as metadata, subprocess, sys
from packaging.version import InvalidVersion, Version

TORCHAO_MINIMUM = Version("0.16.0")


def remove_incompatible_torchao() -> None:
    try:
        installed = metadata.version("torchao")
    except metadata.PackageNotFoundError:
        print("torchao tidak terpasang — aman.")
        return
    try:
        incompatible = Version(installed) < TORCHAO_MINIMUM
    except InvalidVersion:
        incompatible = True
    if not incompatible:
        print("torchao kompatibel:", installed)
        return
    print(f"torchao {installed} tidak kompatibel dengan PEFT; menghapusnya…")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    importlib.invalidate_caches()
    try:
        metadata.version("torchao")
    except metadata.PackageNotFoundError:
        print("torchao lama dihapus; backend PyTorch biasa akan dipakai.")
    else:
        raise RuntimeError("torchao lama masih terdeteksi. Restart sesi Kaggle lalu Run All.")


remove_incompatible_torchao()

# Versi disamakan dengan dependensi BatikCraft Studio agar LoRA hasil latihan
# dijamin dapat dimuat aplikasi.
PACKAGES = [
    "diffusers==0.39.0",
    "transformers>=4.48,<5",
    "accelerate>=1.2,<2",
    "peft==0.17.1",
    "safetensors>=0.8",
    "datasets>=3.2,<5",
    "bitsandbytes>=0.43",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *PACKAGES], check=True)
importlib.invalidate_caches()

for module in ("diffusers", "transformers", "accelerate", "peft", "safetensors"):
    print(f"{module:14s} {metadata.version(module)}")

assert Version(metadata.version("peft")) >= Version("0.17"), (
    "peft terlalu lama untuk diffusers 0.39 — restart sesi lalu Run All."
)
try:
    assert Version(metadata.version("torchao")) >= TORCHAO_MINIMUM
except metadata.PackageNotFoundError:
    pass  # tidak terpasang = kondisi yang diinginkan

import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("\ntorch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {vram_gb:.1f} GB")

In [ ]:
# 2) Konfigurasi. Ubah dataset_root ke folder dataset batik Anda.
from dataclasses import dataclass, asdict, field
from pathlib import Path

@dataclass
class CFG:
    # --- data ---
    dataset_root: str = "/kaggle/input/batik-dataset"   # folder berisi gambar batik
    trigger_word: str = "bcr_batikstyle"                # kata pemicu gaya
    resolution: int = 1024                              # 1024 (SDXL) atau 768 bila VRAM sempit
    # --- model dasar ---
    base_model: str = "stabilityai/stable-diffusion-xl-base-1.0"
    vae_fix: str = "madebyollin/sdxl-vae-fp16-fix"      # VAE stabil untuk fp16
    # --- LoRA ---
    lora_rank: int = 32
    lora_alpha: int = 32
    # --- pelatihan ---
    max_steps: int = 1200
    batch_size: int = 1
    grad_accum: int = 4
    learning_rate: float = 1e-4
    seed: int = 1337
    gradient_checkpointing: bool = True
    use_8bit_adam: bool = True
    # --- keluaran ---
    output_dir: str = "/kaggle/working/batikbrew-sdxl-lora"
    model_id: str = "batikcraft-batikbrew-sdxl-v1"
    model_name: str = "BatikCraft BatikBrew SDXL Style"
    author: str = "Balya Rochmadi"

cfg = CFG()

# T4 (±15 GB) tidak cukup untuk latih SDXL 1024 px; turunkan otomatis ke 768.
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb < 16 and cfg.resolution > 768:
    cfg.resolution = 768
    print(f"VRAM {vram_gb:.1f} GB terdeteksi -> resolusi latih diturunkan ke 768 px.")
if vram_gb < 12:
    cfg.resolution = min(cfg.resolution, 640)
    print("VRAM sangat terbatas -> resolusi 640 px; hasil tetap dapat dipakai.")

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
import json
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
# 3) Dataset: kumpulkan gambar batik + caption bergaya (objek tidak diperlukan).
import json, random
from pathlib import Path
from PIL import Image, ImageFile, ImageOps, UnidentifiedImageError

ImageFile.LOAD_TRUNCATED_IMAGES = True
EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

STYLE_PHRASES = [
    "batik canting motif, hand drawn wax resist lines, isen-isen dots and fills",
    "traditional Indonesian batik ornament, dense organic curves, soga brown palette",
    "batik tulis pattern, fine outline, indigo and cream palette, flat shading",
    "javanese batik ornament, symmetrical flourishes, cecek dotting texture",
]

def build_dataset(root: str, out_dir: str, trigger: str, size: int) -> int:
    source = Path(root)
    target = Path(out_dir) / "train"
    target.mkdir(parents=True, exist_ok=True)
    records, kept, skipped = [], 0, 0

    for path in sorted(source.rglob("*")):
        if path.suffix.lower() not in EXTENSIONS:
            continue
        try:
            with Image.open(path) as raw:
                raw.load()
                image = ImageOps.exif_transpose(raw).convert("RGB")
        except (UnidentifiedImageError, OSError, ValueError):
            skipped += 1
            continue
        # Center-crop lalu resize ke resolusi latih.
        short = min(image.size)
        left = (image.width - short) // 2
        top = (image.height - short) // 2
        image = image.crop((left, top, left + short, top + short)).resize((size, size), Image.LANCZOS)
        name = f"{kept:05d}.jpg"
        image.save(target / name, quality=95)
        caption = f"{trigger}, {random.choice(STYLE_PHRASES)}"
        records.append({"file_name": name, "text": caption})
        kept += 1

    with (target / "metadata.jsonl").open("w", encoding="utf-8") as handle:
        for row in records:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Gambar dipakai: {kept} · dilewati (rusak/format): {skipped}")
    print("Contoh caption:", records[0]["text"] if records else "-")
    return kept

random.seed(cfg.seed)
total_images = build_dataset(cfg.dataset_root, cfg.output_dir, cfg.trigger_word, cfg.resolution)
assert total_images >= 20, "Sediakan minimal ~20 gambar batik agar gaya terbentuk."

In [ ]:
# 4) Pelatihan LoRA SDXL (UNet attention). Hemat VRAM: fp16 + checkpointing + 8-bit Adam.
import gc, math, itertools, torch, torch.nn.functional as F
from accelerate import Accelerator
from datasets import load_dataset
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel, StableDiffusionXLPipeline
from peft import LoraConfig, get_peft_model_state_dict
from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer
from torchvision import transforms

torch.manual_seed(cfg.seed)
accelerator = Accelerator(mixed_precision="fp16", gradient_accumulation_steps=cfg.grad_accum)
device, weight_dtype = accelerator.device, torch.float16

# --- komponen SDXL ---
tokenizer_one = CLIPTokenizer.from_pretrained(cfg.base_model, subfolder="tokenizer")
tokenizer_two = CLIPTokenizer.from_pretrained(cfg.base_model, subfolder="tokenizer_2")
text_encoder_one = CLIPTextModel.from_pretrained(cfg.base_model, subfolder="text_encoder", dtype=weight_dtype).to(device)
text_encoder_two = CLIPTextModelWithProjection.from_pretrained(cfg.base_model, subfolder="text_encoder_2", dtype=weight_dtype).to(device)
vae = AutoencoderKL.from_pretrained(cfg.vae_fix, torch_dtype=torch.float32).to(device)   # VAE fp32 agar stabil
unet = UNet2DConditionModel.from_pretrained(cfg.base_model, subfolder="unet", torch_dtype=weight_dtype).to(device)
noise_scheduler = DDPMScheduler.from_pretrained(cfg.base_model, subfolder="scheduler")

for module in (text_encoder_one, text_encoder_two, vae, unet):
    module.requires_grad_(False)

# --- pasang LoRA hanya pada UNet ---
unet.add_adapter(LoraConfig(
    r=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
))
if cfg.gradient_checkpointing:
    unet.enable_gradient_checkpointing()
trainable = [p for p in unet.parameters() if p.requires_grad]
for parameter in trainable:
    parameter.data = parameter.data.to(torch.float32)
print("Parameter LoRA:", sum(p.numel() for p in trainable) / 1e6, "juta")

optimizer_class = torch.optim.AdamW
if cfg.use_8bit_adam:
    try:
        import bitsandbytes as bnb
        optimizer_class = bnb.optim.AdamW8bit
    except Exception as exc:
        print("8-bit Adam tidak tersedia, memakai AdamW biasa:", exc)
optimizer = optimizer_class(trainable, lr=cfg.learning_rate)

# --- data ---
dataset = load_dataset("imagefolder", data_dir=f"{cfg.output_dir}/train", split="train")
to_tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])

def encode_prompt(captions):
    embeds, pooled = [], None
    for tokenizer, encoder in ((tokenizer_one, text_encoder_one), (tokenizer_two, text_encoder_two)):
        tokens = tokenizer(captions, padding="max_length", max_length=tokenizer.model_max_length,
                           truncation=True, return_tensors="pt").input_ids.to(device)
        output = encoder(tokens, output_hidden_states=True)
        if encoder is text_encoder_two:
            pooled = output[0]
        embeds.append(output.hidden_states[-2])
    return torch.cat(embeds, dim=-1), pooled

def collate(batch):
    pixels = torch.stack([to_tensor(item["image"].convert("RGB")) for item in batch])
    return {"pixel_values": pixels, "captions": [item["text"] for item in batch]}

loader = torch.utils.data.DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True,
                                     collate_fn=collate, num_workers=2, drop_last=True)
unet, optimizer, loader = accelerator.prepare(unet, optimizer, loader)

# --- loop ---
step, losses = 0, []
add_time = torch.tensor([[cfg.resolution, cfg.resolution, 0, 0, cfg.resolution, cfg.resolution]],
                        dtype=weight_dtype, device=device)
unet.train()
while step < cfg.max_steps:
    for batch in loader:
        with accelerator.accumulate(unet):
            with torch.no_grad():
                latents = vae.encode(batch["pixel_values"].to(device, dtype=torch.float32)).latent_dist.sample()
                latents = (latents * vae.config.scaling_factor).to(weight_dtype)
                prompt_embeds, pooled_embeds = encode_prompt(batch["captions"])

            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                      (latents.shape[0],), device=device).long()
            noisy = noise_scheduler.add_noise(latents, noise, timesteps)
            prediction = unet(
                noisy, timesteps, prompt_embeds,
                added_cond_kwargs={"text_embeds": pooled_embeds,
                                   "time_ids": add_time.repeat(latents.shape[0], 1)},
            ).sample
            target = noise if noise_scheduler.config.prediction_type == "epsilon" else \
                     noise_scheduler.get_velocity(latents, noise, timesteps)
            loss = F.mse_loss(prediction.float(), target.float(), reduction="mean")
            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.detach().item())
        step += 1
        if step % 25 == 0 or step == 1:
            print(f"step {step}/{cfg.max_steps} · loss {sum(losses[-25:]) / len(losses[-25:]):.4f}")
        if step >= cfg.max_steps:
            break

# --- simpan bobot LoRA dalam format diffusers ---
unet_lora = get_peft_model_state_dict(accelerator.unwrap_model(unet))
StableDiffusionXLPipeline.save_lora_weights(cfg.output_dir, unet_lora_layers=unet_lora, safe_serialization=True)
del vae, text_encoder_one, text_encoder_two
gc.collect(); torch.cuda.empty_cache()
print("Bobot LoRA tersimpan di", cfg.output_dir)

In [ ]:
# 5) Uji cepat: foto objek -> gaya batik (img2img). Ganti OBJECT_PATH dengan foto Anda.
import torch
from PIL import Image
from diffusers import AutoPipelineForImage2Image

OBJECT_PATH = "/kaggle/input/contoh-objek/botol.jpg"   # foto botol/bunga/objek apa pun
STRENGTH = 0.55        # 0.4 menjaga bentuk, 0.7 lebih bergaya batik

pipe = AutoPipelineForImage2Image.from_pretrained(
    cfg.base_model, torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
).to("cuda")
pipe.load_lora_weights(cfg.output_dir)
pipe.set_progress_bar_config(disable=True)

source = Image.open(OBJECT_PATH).convert("RGB").resize((cfg.resolution, cfg.resolution), Image.LANCZOS)
result = pipe(
    prompt=f"{cfg.trigger_word}, batik ornament rendition of the object, canting outline, isen-isen fill",
    negative_prompt="photograph, 3d render, text, watermark, changed silhouette, extra object",
    image=source, strength=STRENGTH, guidance_scale=7.0, num_inference_steps=30,
).images[0]
result.save("/kaggle/working/preview-batik.png")
display(source.resize((384, 384)), result.resize((384, 384)))

## Di mana "pasangan" botol → botol gaya batik?

**Tidak ada pasangan di dataset — dan memang tidak diperlukan.** Ini pelatihan
*style transfer*, bukan *paired translation*:

| Tahap | Yang menyediakan **bentuk** | Yang menyediakan **gaya** |
| --- | --- | --- |
| Latih | — (tidak ada objek) | gambar batik + kata pemicu |
| Inferensi | foto botol Anda (img2img + ControlNet) | LoRA hasil latihan |

Jadi "botol → botol gaya batik" terbentuk saat generate: foto botol menjadi
kerangka, LoRA melukiskan canting dan isen-isen di atasnya. Konsekuensinya satu
LoRA berlaku untuk objek apa pun — botol, bunga, wayang, kursi — tanpa pernah
melihat objek itu saat latihan.

**Dua kunci agar botol tetap berbentuk botol:**

1. `strength` img2img rendah (0,40–0,55) — makin tinggi makin bergaya tapi makin
   jauh dari bentuk asli.
2. **ControlNet Canny** mengunci siluet dari tepi gambar sumber (sel berikutnya).

**Kapan pelatihan berpasangan (paired) benar-benar diperlukan?** Hanya bila Anda
ingin transformasi yang sangat spesifik dan konsisten, misalnya "setiap botol
harus menjadi motif kawung dengan komposisi tertentu". Itu butuh ratusan pasang
gambar (asli + versi batiknya) yang harus Anda buat sendiri — biayanya besar dan
hasilnya justru kurang umum. Alur di bawah lebih praktis: hasilkan kandidat
dengan LoRA gaya, kurasi yang bagus, lalu (opsional) pakai sebagai data
penyempurna.

In [ ]:
# 5b) Inferensi terkunci siluet: ControlNet Canny + LoRA gaya.
#     Inilah yang membuat "botol tetap botol", hanya gayanya yang berubah.
import cv2, numpy as np, torch
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionXLControlNetImg2ImgPipeline

OBJECT_PATH = "/kaggle/input/contoh-objek/botol.jpg"
STRENGTH = 0.55            # 0.40 sangat setia bentuk · 0.70 sangat bergaya
CONTROL_SCALE = 0.8        # kekuatan penguncian siluet

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16
)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
    cfg.base_model, controlnet=controlnet, torch_dtype=torch.float16,
    variant="fp16", use_safetensors=True,
).to("cuda")
pipe.load_lora_weights(cfg.output_dir)
pipe.enable_model_cpu_offload()          # aman untuk GPU 16 GB
pipe.set_progress_bar_config(disable=True)

source = Image.open(OBJECT_PATH).convert("RGB").resize(
    (cfg.resolution, cfg.resolution), Image.LANCZOS
)
edges = cv2.Canny(np.array(source), 100, 200)
control = Image.fromarray(np.stack([edges] * 3, axis=-1))

result = pipe(
    prompt=f"{cfg.trigger_word}, batik ornament rendition, canting outline, isen-isen fill, flat batik palette",
    negative_prompt="photograph, 3d render, text, watermark, changed silhouette, extra object, background scene",
    image=source, control_image=control,
    strength=STRENGTH, controlnet_conditioning_scale=CONTROL_SCALE,
    guidance_scale=7.0, num_inference_steps=30,
).images[0]
result.save("/kaggle/working/preview-batik-controlnet.png")
display(source.resize((320, 320)), control.resize((320, 320)), result.resize((320, 320)))

In [ ]:
# 5c) OPSIONAL — bangun "pasangan" asli↔batik untuk dokumentasi/kurasi.
#     Berguna untuk menilai konsistensi, membuat contoh di marketplace, atau
#     sebagai bahan penyempurnaan lanjutan. Bukan syarat pelatihan gaya.
from pathlib import Path

OBJECT_FOLDER = "/kaggle/input/contoh-objek"     # folder berisi foto objek
PAIR_OUTPUT = Path("/kaggle/working/pairs")
PAIR_OUTPUT.mkdir(parents=True, exist_ok=True)

photos = sorted(
    p for p in Path(OBJECT_FOLDER).rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)[:12]

manifest_rows = []
for index, photo in enumerate(photos):
    original = Image.open(photo).convert("RGB").resize(
        (cfg.resolution, cfg.resolution), Image.LANCZOS
    )
    edges = cv2.Canny(np.array(original), 100, 200)
    control = Image.fromarray(np.stack([edges] * 3, axis=-1))
    styled = pipe(
        prompt=f"{cfg.trigger_word}, batik ornament rendition, canting outline, isen-isen fill",
        negative_prompt="photograph, text, watermark, changed silhouette, extra object",
        image=original, control_image=control,
        strength=STRENGTH, controlnet_conditioning_scale=CONTROL_SCALE,
        guidance_scale=7.0, num_inference_steps=30,
    ).images[0]

    original.save(PAIR_OUTPUT / f"{index:03d}_asli.png")
    styled.save(PAIR_OUTPUT / f"{index:03d}_batik.png")
    # Tempelkan berdampingan untuk perbandingan cepat.
    side = Image.new("RGB", (cfg.resolution * 2, cfg.resolution), "white")
    side.paste(original, (0, 0)); side.paste(styled, (cfg.resolution, 0))
    side.resize((768, 384)).save(PAIR_OUTPUT / f"{index:03d}_perbandingan.jpg", quality=90)
    manifest_rows.append({"source": photo.name, "original": f"{index:03d}_asli.png",
                          "batik": f"{index:03d}_batik.png"})

(PAIR_OUTPUT / "pairs.json").write_text(
    json.dumps(manifest_rows, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"{len(manifest_rows)} pasangan tersimpan di {PAIR_OUTPUT}")

In [ ]:
# 6) Bungkus menjadi paket .batikmodel yang siap dipasang di BatikCraft Studio.
import hashlib, json, zipfile
from pathlib import Path

def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

weights = Path(cfg.output_dir) / "pytorch_lora_weights.safetensors"
assert weights.is_file(), "Bobot LoRA belum ada — jalankan sel pelatihan terlebih dahulu."

manifest = {
    "format": "batikcraft-model-pack",
    "schema_version": "1.0",
    "model": {
        "id": cfg.model_id,
        "name": cfg.model_name,
        "version": "1.0.0",
        "type": "lora",
        # PENTING: sdxl -> BatikCraft memakai pipeline SDXL untuk model ini.
        "base_model_family": "sdxl",
        "trigger_words": [cfg.trigger_word],
        "recommended_weight": 0.85,
        "resolution": cfg.resolution,
        "capabilities": ["object-batification", "img2img", "preserve-silhouette", "batikbrew"],
        "lora_file": "model/pytorch_lora_weights.safetensors",
        "author": cfg.author,
        "description": "LoRA gaya batik SDXL untuk mengubah objek/foto apa pun menjadi ornamen batik.",
        "negative_prompt": "photograph, 3d render, text, watermark, changed silhouette, extra object",
        "metadata": {
            "trained_steps": cfg.max_steps,
            "lora_rank": cfg.lora_rank,
            "learning_rate": cfg.learning_rate,
            "training_images": int(total_images),
            "sha256": sha256_of(weights),
        },
    },
}

package_path = Path("/kaggle/working") / f"{cfg.model_id}.batikmodel"
with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("manifest.json", json.dumps(manifest, indent=2, ensure_ascii=False))
    archive.write(weights, "model/pytorch_lora_weights.safetensors")

print("Paket siap:", package_path, f"({package_path.stat().st_size / 1024**2:.1f} MB)")
print("Unduh dari panel Output, lalu pasang lewat:")
print("  Pusat Dependensi -> tab Model AI Offline & LoRA -> Pasang .batikmodel…")